In [1]:
%reset -f
from pydantic import * # type: ignore
from pprint import pprint
from datetime import datetime

In [2]:
class Cookie(BaseModel):
    flavor: str
    calories: int

There are more ways to configure the model when using Python's builtin `dataclass` or `TypedDict`. But my current thinking is that I should not mix `dataclass` and `BaseModel`, it might make debugging down the line harder. And using `TypedDict` is just a bad code smell.

I can use model config on type adapters as well. But the example I found in the documentation was pretty lame. And I have not been able to come up with a better example by myself so far.

In [46]:
Strings = TypeAdapter(list[str], config=ConfigDict(coerce_numbers_to_str=True))
vals = Strings.validate_python([1, "2", 3.14])
vals

['1', '2', '3.14']

## Generic Models

Works pretty much like generic types in Python itself. Supports the new syntax and all. When using bounded types, make sure to specify the type parameter when instantiating the data object. If done incorrectly, it might lead to data loss as shown below.

In [86]:
%reset -f
from pydantic import *  # type: ignore

In [87]:
class Cookie(BaseModel):
    flavor: str
    calories: int


class Snack[T: Cookie](BaseModel):
    cookie: T
    serving_size: int


class LowFatCookie(Cookie):
    carbs_gms: int

In [88]:
# OK: Instantiating with the correct type parameter set to LowFatCookie.
Snack[LowFatCookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100, 'carbs_gms': 25},
 'serving_size': 2}

In [89]:
# OK: Even when the type parameter is left out, it will still work because there are on wonky default types.
Snack(
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100, 'carbs_gms': 25},
 'serving_size': 2}

In [90]:
# Type param is Cookie even though the data ha LowFatCookie. carbs_gms is lost.
Snack[Cookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100}, 'serving_size': 2}

## Dynamic Model Creation

Creates the model class at runtime. Here is a simple example. More advanced examples in the [docs](https://docs.pydantic.dev/latest/concepts/models/#dynamic-model-creation).

In [91]:
%reset -f
from pydantic import *  # type: ignore

In [92]:
Cookie = create_model("Cookie", calories=int, flavor=(str, "Chocolate Chip"))
Cookie

__main__.Cookie

The above code is equivalent to -
```python
class Cookie(BaseModel):
    calories: int
    flaovr: str = "Chocolate Chip"
```

In [93]:
Cookie(calories=200)

Cookie(calories=200, flavor='Chocolate Chip')

## Root models
If I want to define a Pydantic data model made up of Python types without going through the hassle of authoring a class and attributes, etc. I can use root models.

In [94]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint
from typing import Iterator

In [95]:
Hobbies = RootModel[list[str]]

In [96]:
hobbies = Hobbies(["Car Racing", "Video Games"])
hobbies

RootModel[list[str]](root=['Car Racing', 'Video Games'])

In [97]:
pprint(hobbies.model_dump())
pprint(hobbies.model_dump_json())

['Car Racing', 'Video Games']
'["Car Racing","Video Games"]'


In [98]:
# But I cannot index into it.
try:
    hobbies[0]
except TypeError as err:
    pprint(err)

TypeError("'RootModel[list[str]]' object is not subscriptable")


In [99]:
# If I want better semantics like indexing I need to implement my own __getitem__ and __iter__ methods
class StockTicker(RootModel):
    root: dict[str, float]

    def __getitem__(self, idx: str) -> float:
        return self.root[idx]

    def __iter__(self) -> Iterator[str]:
        yield from self.root.keys()

In [100]:
ticker = StockTicker({"APPL": 200, "MSFT": 240})

In [101]:
ticker["APPL"]

200.0

In [102]:
for stock in ticker:
    print(stock, ticker[stock])

APPL 200.0
MSFT 240.0


## Automatically Excluded Attributes

Model attributes marked with a `ClassVar` or having their name start with a single underscore are excluded from the schema. The private instance attributes are not even created during `__init__` and there is no way I can access it from either within or outside the model. The only way to use them is to first intialize them in the `model_post_init` method.

In [103]:
%reset -f
from pydantic import *  # type: ignore
from typing import ClassVar, Any

In [104]:
class User(BaseModel):
    connstr: ClassVar[str] = "sqlite://exampledb"
    # Without the model_post_init this is useless
    _partition_id: int
    id: int
    name: str

In [105]:
User.model_json_schema()

{'properties': {'id': {'title': 'Id', 'type': 'integer'},
  'name': {'title': 'Name', 'type': 'string'}},
 'required': ['id', 'name'],
 'title': 'User',
 'type': 'object'}

In [106]:
user = User(_partition_id=0, id=1, name="APTG")
user

User(id=1, name='APTG')

In [107]:
user.connstr

'sqlite://exampledb'

In [108]:
User.connstr

'sqlite://exampledb'

In [109]:
try:
    user._partition_id
except AttributeError as err:
    print(err)

'User' object has no attribute '_partition_id'


In [110]:
class Photo(BaseModel):
    connstr: ClassVar[str] = "sqlite://exampledb"
    _partition_id: int

    id: int
    title: str

    def model_post_init(self, context: Any) -> None:
        self._partition_id = self.id % 10

In [111]:
photo = Photo(id=1, title="Vacation")  # type: ignore

In [112]:
photo._partition_id

1

## Default Values
As far as I can tell, there are three ways to set default values -
  * With the `field_name: type = default_val` declaration
  * Specifying in the `Field` function `field_name: type = Field(default=default_val)`
  * Specifying a default factory in the `Field` function `field_name: type = Field(default_factory: func)`

Just like in normal Python, don't use mutable objects like empty lists or dicts as defaults. Even though Pydantic will automatically do the right thing by making copies and what not, it is just a bad code smell.  

In [119]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint
import random
from uuid import uuid4

In [120]:
class Cookie(BaseModel):
    flavor: str = Field(default="Chocolate Chip")
    calories: int = Field(default_factory=lambda: random.randint(100, 200))

In [121]:
Cookie()

Cookie(flavor='Chocolate Chip', calories=118)

The default factory can also take a single required argument, in which case the already validated data will be passed as a dictionary.

In [122]:
class User(BaseModel):
    id: str = Field(default_factory=lambda: uuid4().hex)
    email: str | None = None
    name: str = Field(
        default_factory=lambda data: data["email"] if data["email"] else "Anon"
    )

In [123]:
User()

User(id='4e7e3ea4e8e54e06a68cc6bcb36e3a8e', email=None, name='Anon')

In [124]:
User(email="cookie@monster.com")

User(id='520bf72910e240808dd31161c8c7c019', email='cookie@monster.com', name='cookie@monster.com')

In [125]:
User(email="cookie@monster.com", name="Cookie Monster")

User(id='2773726dd1de4a19836057bccdde590e', email='cookie@monster.com', name='Cookie Monster')

The order of declaration of fields is important for this to work. If `name` had been defined before `email`, the input `data` would not have included `email`!

By default, default values are not validated. I can change this behavior either with a field argument or a with a model config.

In [126]:
class Cookie(BaseModel):
    calories: int = "hello"
    flavor: str

    model_config = ConfigDict(validate_default=True)

In [127]:
try:
    Cookie(flavor="Chocolate Chip")
except ValidationError as err:
    pprint(err.errors())

[{'input': 'hello',
  'loc': ('calories',),
  'msg': 'Input should be a valid integer, unable to parse string as an '
         'integer',
  'type': 'int_parsing',
  'url': 'https://errors.pydantic.dev/2.11/v/int_parsing'}]


In [128]:
class Rectangle(BaseModel):
    length: float
    breadth: int = Field(
        default_factory=lambda data: data["length"], validate_default=True
    )

In [129]:
Rectangle(length=10)

Rectangle(length=10.0, breadth=10)

In [130]:
try:
    Rectangle(length=10.1)
except ValidationError as err:
    pprint(err.errors())

[{'input': 10.1,
  'loc': ('breadth',),
  'msg': 'Input should be a valid integer, got a number with a fractional part',
  'type': 'int_from_float',
  'url': 'https://errors.pydantic.dev/2.11/v/int_from_float'}]


---

# Appendix

## Misc Fields Support
There are a bunch of other things I can do with the `Field` function like controlling how or whether it will be outputted, marking them for deprecation, etc. 

## Alias Details
Some more details on alias in case I have to use it.

I can have different names for my fields. Here are some use cases I can think for these -
  * Accomodate different naming conventions across different systems
  * Seperate out the internal data model from the application model (?)

The default "alias" is implemented in a weird way. When I am instantiating the model I have to use the alias, but when accessing that field on the data object, I have to use the "real" name of the field. When the model is serialized, again the real name is used.

I think in order to compensate for this design choice, Pydantic has introduced the concept of seralization and validation aliases. Serialization alias is to be used for accessing the fields on the data object and when the model is serialized. Validation alias is to be used when instantiating the model.

In [167]:
class User(BaseModel):
    name: str = Field(alias="username")

In [168]:
# Use the alias at initialization
user = User(username="APTG")
user

User(name='APTG')

In [169]:
# Serialization will use the real name
user.model_dump()

{'name': 'APTG'}

In [170]:
# Field access is also via the real name
user.name

'APTG'

In [171]:
# Trying to access the alias will throw a plain old Python error
try:
    user.username
except AttributeError as err:
    print(err)

'User' object has no attribute 'username'


In [172]:
# Trying to initialize the model with the real name will throw a Pydantic error
try:
    User(name="APTG")
except ValidationError as err:
    print(err.errors())

[{'type': 'missing', 'loc': ('username',), 'msg': 'Field required', 'input': {'name': 'APTG'}, 'url': 'https://errors.pydantic.dev/2.11/v/missing'}]


Looks like validation alias and alias are semantically similar!

In [173]:
%reset -f
from pydantic import *
from pprint import pprint

In [174]:
class User(BaseModel):
    name: str = Field(validation_alias="username")

In [175]:
# Use the alias at initialization
# PyLance is giving a false positive!
user = User(username="APTG")  # type: ignore
user

User(name='APTG')

In [176]:
# Serialization will use the real name
user.model_dump()

{'name': 'APTG'}

In [177]:
# Field access is also via the real name
user.name

'APTG'

In [178]:
# Trying to access the alias will throw a plain old Python error
try:
    user.username
except AttributeError as err:
    print(err)

'User' object has no attribute 'username'


In [179]:
# Trying to initialize the model with the real name will throw a Pydantic error
try:
    User(name="APTG")
except ValidationError as err:
    pprint(err.errors())

[{'input': {'name': 'APTG'},
  'loc': ('username',),
  'msg': 'Field required',
  'type': 'missing',
  'url': 'https://errors.pydantic.dev/2.11/v/missing'}]


The difference is that I can have multiple validation aliases.

In [180]:
%reset -f
from pydantic import *
from pprint import pprint

In [181]:
class User(BaseModel):
    # I am going to include the real name in the alias for consistency
    name: str = Field(validation_alias=AliasChoices("name", "username", "login"))

In [182]:
# Now I can instantiate the model with the aliased real name
user = User(name="APTG")
user

User(name='APTG')

In [183]:
user.name

'APTG'

In [184]:
try:
    user.username
except AttributeError as err:
    print(err)

'User' object has no attribute 'username'


In [185]:
User(username="Avilay")  # type: ignore

User(name='Avilay')

Now lets try out serialization alias. By default serialization will **not** use the serialization alias! I have to specify a `by_alias` flag during serialization for this to happen. This flag can be set on a per call basis everytime I serialize, or it can be set in the model config.

In [186]:
class Song(BaseModel):
    name: str = Field(serialization_alias="title")

    # model_config = ConfigDict(serialize_by_alias=True)

In [187]:
song = Song(name="Abracadabra")
song

Song(name='Abracadabra')

In [188]:
song.model_dump()

{'name': 'Abracadabra'}

In [189]:
song.model_dump(by_alias=True)

{'title': 'Abracadabra'}